# Head-to-head MCTS: production vs sidekick-guided search

Two search engines play each other. Both use the **same value network** and the **same MCTS code and settings as production**
(`engine/mcts.py`: PUCT `c = 1.10`, top-10 policy priors, softmax `tau = 1`, most-visited move, checkmate value 1.03; verified
against the production `MCTS` and `predict_next_move` outputs).

| Engine | Policy at the root | Policy at every other expansion | Simulations |
|---|---|---|---|
| **prod** | base (28M) | base (28M) | 400 (production hard mode) |
| **sidekick** | base (28M) | sidekick (4.1M) | set per matchup |

## Upload

One Kaggle Dataset with these files (any folder layout; the notebook searches `/kaggle/input`):

| Local path | Upload as |
|---|---|
| `policy_transformer/ultra_3o9.pt` | `ultra_3o9.pt` (base policy) |
| `policy_transformer/best_sidekick_v1.pt` | `best_sidekick_v1.pt` (sidekick policy) |
| `value_transformer/mini_value_6o4.pt` | `mini_value_6o4.pt` (value network) |
| `policy_transformer/vocab.pkl` | `vocab.pkl` (base vocab, 4033) |
| `policy_transformer/vocab1.pkl` | `vocab1.pkl` (sidekick vocab, 1793) |

## Run

1. Settings: Accelerator **GPU T4**, Internet **on** (only used for `pip install chess` if it is missing).
2. Add Data, attach the dataset.
3. **Save Version, Save & Run All (Commit)**. It runs in the background, so you can close the browser.
4. The last cell has the settings: 10 openings, both matchups, and a time budget of 8 h (Kaggle's GPU limit is 9 h, which leaves a margin for the game in progress and the report;
   no new opening is started after a matchup's share of the budget, so games always come in complete colour pairs).
5. Results are in the version's **Output** tab: `h2h_summary.json`, `h2h_<matchup>.json`, `h2h_<matchup>.pgn`.
   They are rewritten after every opening, so a stopped run keeps what finished.

## Design

* **Search is deterministic**, so one game per colour from the start position is a sample of size 2. Each game instead starts from a
  different opening (the first 10 of 16 built-in book lines, 6 to 8 plies) and every opening is played twice with colours swapped: 20 games per matchup.
* **Equal compute.** `equal_time`: the notebook measures seconds per move for both engines on the same positions on the Kaggle GPU,
  then sets the sidekick's simulation count so its mean time per move matches prod's (a fixed count afterwards, so games are
  reproducible). `equal_sims`: 400 vs 400, which isolates move quality from speed.
* Both engines share one implementation of the policy call, so timing differences come from model size, not code.
* The self-test and calibration run first and print how much faster the sidekick really is per move. If it is barely faster
  (the value network and board handling cost about as much as the policy call), the two matchups will be nearly the same.
* Games end by checkmate, stalemate, insufficient material, a claimable draw, or a 300-ply cap (adjudicated draw).
* The final cell prints, per matchup: W/D/L, score with a bootstrap 95% CI over openings, approximate Elo difference,
  seconds per move, and policy calls per move for each engine.


In [ ]:
%%writefile policy_model.py
import torch
import torch.nn as nn
from torch.nn import functional as F
import math


def _apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply rotary position embedding to a tensor shaped [B, n_head, T, head_dim].
    cos/sin are shaped [T, head_dim//2] (broadcasted inside).
    """
    # x: [..., head_dim] where head_dim is even
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    out_even = x_even * cos - x_odd * sin
    out_odd = x_even * sin + x_odd * cos
    return torch.stack((out_even, out_odd), dim=-1).flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        # output projection
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        head_dim = n_embd // n_head
        assert head_dim % 2 == 0, "RoPE requires an even head dimension"

        # RoPE (rotary position embeddings) cache (non-persistent so old checkpoints still load)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("_rope_inv_freq", inv_freq, persistent=False)
        self.register_buffer("_rope_cos_cached", torch.empty(0), persistent=False)
        self.register_buffer("_rope_sin_cached", torch.empty(0), persistent=False)
        self._rope_seq_len_cached = 0
        self._rope_cache_dtype = None
        self._rope_cache_device = None

        # Flash Attention requires PyTorch 2.0+
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: Flash Attention not available. Using slow manual attention.")
            self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                        .view(1, 1, block_size, block_size))

    def _rope_cos_sin(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> tuple[torch.Tensor, torch.Tensor]:
        # Cache per (device, dtype, seq_len>=T). Compute in fp32 for stability, cast for use.
        if (
            self._rope_seq_len_cached >= seq_len
            and self._rope_cache_device == device
            and self._rope_cache_dtype == dtype
            and self._rope_cos_cached.numel() != 0
        ):
            return self._rope_cos_cached[:seq_len], self._rope_sin_cached[:seq_len]

        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        freqs = torch.outer(t, self._rope_inv_freq.to(device=device))  # [T, head_dim//2]
        cos = freqs.cos().to(dtype=dtype)
        sin = freqs.sin().to(dtype=dtype)

        self._rope_cos_cached = cos
        self._rope_sin_cached = sin
        self._rope_seq_len_cached = seq_len
        self._rope_cache_device = device
        self._rope_cache_dtype = dtype
        return cos, sin

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # RoPE: apply rotary embeddings to q/k (not v)
        cos, sin = self._rope_cos_sin(T, device=q.device, dtype=q.dtype)
        q = _apply_rope(q, cos, sin)
        k = _apply_rope(k, cos, sin)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True
            )
        else:
            # manual implementation (fallback)
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        y = self.resid_dropout(self.c_proj(y))
        return y

class FeedFoward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedFoward(n_embd, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class PolicyHead(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_head, n_layer, dropout, device):
        super().__init__()
        self.block_size = block_size
        self.device = device
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # Ensure T is within block_size
        if T > self.block_size:
            idx = idx[:, -self.block_size:]
            T = self.block_size
            if targets is not None:
                targets = targets[:, -self.block_size:]

        tok_emb = self.token_embedding_table(idx) 
        # With RoPE applied inside attention, we don't add absolute positional embeddings here.
        x = tok_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        if targets is None:
            logits = self.lm_head(x[:, [-1], :]) # optimization: only compute last logit for inference
            loss = None
        else:
            logits = self.lm_head(x)
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=-100) # Added ignore_index explicitly
        
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
%%writefile value_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint # optimization

# -----------------------------------------------------------------------------
# OPTIMIZED ATTENTION (With Relative Position Bias)
# -----------------------------------------------------------------------------
class SelfAttention(nn.Module):
    def __init__(self, n_embed, n_heads, block_size, dropout):
        super().__init__()
        assert n_embed % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = n_embed // n_heads
        self.dropout = dropout

        self.c_attn = nn.Linear(n_embed, 3 * n_embed, bias=False)
        self.c_proj = nn.Linear(n_embed, n_embed)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # Flash check (same as policy)
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            print("Flash Attention not available - using manual.")
            self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # B nh T hd
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0.0, is_causal=False  # No causal for value model
            )
        else:
            att = torch.matmul(q, k.transpose(-2, -1)) * (self.head_dim ** -0.5)
            att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = torch.matmul(att, v)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y
# -----------------------------------------------------------------------------
# FEED FORWARD
# -----------------------------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(), 
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------------------------------------------------------
# TRANSFORMER BLOCK (With Gradient Checkpointing)
# -----------------------------------------------------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, n_embed, n_heads, block_size, dropout):
        super().__init__()
        self.sa = SelfAttention(n_embed, n_heads, block_size, dropout)
        self.ff = FeedForward(n_embed, dropout)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        # Standard Residual Connection
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# -----------------------------------------------------------------------------
# VALUE HEAD
# -----------------------------------------------------------------------------
class ValueHead(nn.Module):
    def __init__(self, n_embed, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, n_embed // 2),
            nn.GELU(),
            nn.Linear(n_embed // 2, 1),
            nn.Tanh(), 
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = x.mean(dim=1) 
        out = self.dropout(out)
        out = self.net(out)
        return out

# -----------------------------------------------------------------------------
# CHESSFORMER (Value Only - Optimized)
# -----------------------------------------------------------------------------
class ChessFormer(nn.Module):
    def __init__(self, block_size=64, n_embed=256, n_heads=4, n_layers=6, dropout=0.1):  # Scaled down: embed 512→256, heads 8→4, layers 8→6 (~8M params)
        super().__init__()  
        self.block_size = block_size
        self.n_embed = n_embed
        
        # Embeddings
        self.piece_embedding = nn.Embedding(13, n_embed) 
        self.rank_embedding = nn.Embedding(8, n_embed)   
        self.file_embedding = nn.Embedding(8, n_embed)   
        self.pos_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            TransformerBlock(n_embed, n_heads, block_size, dropout) 
            for _ in range(n_layers)
        ])
        
        self.ln_f = nn.LayerNorm(n_embed)
        self.value_head = ValueHead(n_embed, dropout)
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, x):
        B, T = x.shape
        device = x.device
        
        ranks = torch.arange(64, device=device) // 8
        files = torch.arange(64, device=device) % 8
        ranks = ranks.unsqueeze(0).expand(B, -1)
        files = files.unsqueeze(0).expand(B, -1)
        
        x = self.piece_embedding(x) + self.rank_embedding(ranks) + self.file_embedding(files)
        x = self.pos_drop(x)
        
        # Loop with Gradient Checkpointing for memory efficiency
        for block in self.blocks:
            if self.training:
                # Use checkpointing during training to save VRAM
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
                
        x = self.ln_f(x)
        value = self.value_head(x)
        
        return value

In [ ]:
import argparse
import json
import math
import os
import pickle
import subprocess
import sys
import time

try:
    import chess
except ImportError:                       # needs Internet on in the Kaggle session settings
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chess"], check=True)
    import chess
import chess.pgn
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

import glob
from policy_model import PolicyHead     # written by the two %%writefile cells above
from value_model import ChessFormer     # (copies of policy_transformer/model.py and value_transformer/model.py)

SEARCH_ROOTS = ["/kaggle/input"] + [p for p in os.environ.get("MODEL_DIRS", "").split(":") if p] + ["."]


def find_file(name):
    for root in SEARCH_ROOTS:
        hits = sorted(glob.glob(os.path.join(root, "**", name), recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found under {SEARCH_ROOTS}. Attach the dataset that contains it.")


# ── Checkpoints ──
BASE_CKPT = find_file("ultra_3o9.pt")
SIDEKICK_CKPT = find_file("best_sidekick_v1.pt")
VALUE_CKPT = find_file("mini_value_6o4.pt")
BASE_VOCAB = find_file("vocab.pkl")
SIDEKICK_VOCAB = find_file("vocab1.pkl")
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "h2h_results"

# ── Architectures (must match the checkpoints) ──
BASE_CFG = dict(n_embd=500, n_head=10, n_layer=8)
SIDEKICK_CFG = dict(n_embd=256, n_head=8, n_layer=4)
BLOCK_SIZE = 128

# ── Production search settings (engine/mcts.py defaults; app.py /v1/api/mcts-3, hard mode) ──
PROD_SIMS = 400
C_PUNC = 1.10
TOP_K = 10
TAU = 1.0

# ── Experiment ──
MAX_PLIES = 300         # longer games are adjudicated as draws
CALIB_POSITIONS = 12    # positions used to measure time per move
MAX_SIM_MULT = 4.0      # cap on the calibrated simulation count (x PROD_SIMS)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

BOOK = [
    "e4 e5 Nf3 Nc6 Bb5 a6",             # Ruy Lopez
    "e4 e5 Nf3 Nc6 Bc4 Bc5",            # Italian
    "e4 c5 Nf3 d6 d4 cxd4 Nxd4 Nf6",    # Sicilian, Najdorf move order
    "e4 e6 d4 d5 Nc3 Nf6",              # French
    "e4 c6 d4 d5 Nc3 dxe4 Nxe4 Bf5",    # Caro-Kann
    "d4 d5 c4 e6 Nc3 Nf6",              # Queen's Gambit Declined
    "d4 Nf6 c4 g6 Nc3 Bg7 e4 d6",       # King's Indian
    "d4 Nf6 c4 e6 Nc3 Bb4",             # Nimzo-Indian
    "c4 e5 Nc3 Nf6 Nf3 Nc6",            # English
    "Nf3 d5 g3 Nf6 Bg2 e6",             # Reti
    "d4 d5 Nf3 Nf6 Bf4 e6",             # London
    "e4 d5 exd5 Qxd5 Nc3 Qa5",          # Scandinavian
    "e4 e5 Nf3 Nf6 Nxe5 d6 Nf3 Nxe4",   # Petrov
    "d4 Nf6 c4 e6 Nf3 b6",              # Queen's Indian
    "d4 d5 c4 c6 Nf3 Nf6 Nc3 dxc4",     # Slav
    "e4 g6 d4 Bg7 Nc3 d6",              # Modern / Pirc
]


# ── Models and the two functions the search calls ──
def strip_compile_prefix(sd):
    return {k.replace("_orig_mod.", ""): v for k, v in sd.items()}


def load_policy(ckpt_path, vocab_path, cfg):
    with open(vocab_path, "rb") as f:
        stoi = pickle.load(f)
    model = PolicyHead(vocab_size=len(stoi), block_size=BLOCK_SIZE, dropout=0.0, device=DEVICE, **cfg)
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    model.load_state_dict(strip_compile_prefix(sd))
    model.to(DEVICE).eval()
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"policy {os.path.basename(ckpt_path)}: {n:.2f}M params, vocab {len(stoi)}")
    return model, stoi


def mirror_uci(uci):
    # 'e2e4' -> 'e7e5' (ranks flipped); same as policy_transformer/inference.py
    return f"{uci[0]}{9 - int(uci[1])}{uci[2]}{9 - int(uci[3])}{uci[4:]}"


class PolicyFn:
    """Same output as policy_transformer.inference.predict_next_move: the top_k legal moves as
    (logit, uci), sorted by logit, with the history mirrored when Black is to move. The history
    comes in as a UCI move list instead of a PGN string (no re-parsing), and the logits of the
    legal moves are read in one transfer. `calls` counts forward passes."""

    def __init__(self, model, stoi, name):
        self.model, self.stoi, self.name, self.calls = model, stoi, name, 0
        self.pad = stoi.get("|", 0)

    @torch.no_grad()
    def __call__(self, moves, board, top_k):
        self.calls += 1
        mirror = board.turn == chess.BLACK
        if len(moves) == 0:
            ids = [0]
        else:
            ids = [self.stoi.get((mirror_uci(u) if mirror else u)[:4], self.pad) for u in moves]
        idx = torch.tensor([ids[-BLOCK_SIZE:]], dtype=torch.long, device=DEVICE)
        last = self.model(idx)[0][0, -1, :]

        legal = list(board.legal_moves)
        if not legal:
            return []
        ucis, tok_ids = [], []
        for m in legal:
            u = m.uci()
            ucis.append(u)
            tok_ids.append(self.stoi.get((mirror_uci(u) if mirror else u)[:4], -1))
        sel = torch.tensor([max(t, 0) for t in tok_ids], dtype=torch.long, device=DEVICE)
        scores = last.index_select(0, sel).float().cpu().tolist()
        cand = [(s if t >= 0 else float("-inf"), u) for s, t, u in zip(scores, tok_ids, ucis)]
        cand.sort(key=lambda x: x[0], reverse=True)
        return cand[:top_k]


class ValueFn:
    """Same as value_transformer.inference.ChessEvaluator.batch_evaluate (FEN list -> values)."""
    PIECE_MAP = {'.': 0, 'P': 1, 'N': 2, 'B': 3, 'R': 4, 'Q': 5, 'K': 6,
                 'p': 7, 'n': 8, 'b': 9, 'r': 10, 'q': 11, 'k': 12}

    def __init__(self, ckpt_path):
        # mini_value_6o4: 384 embd, 6 heads, 6 layers (production inference.py config)
        self.model = ChessFormer(block_size=64, n_embed=384, n_heads=6, n_layers=6, dropout=0.0)
        sd = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
        self.model.load_state_dict(strip_compile_prefix(sd))
        self.model.to(DEVICE).eval()
        self.calls = 0
        n = sum(p.numel() for p in self.model.parameters()) / 1e6
        print(f"value {os.path.basename(ckpt_path)}: {n:.2f}M params")

    def _tokens(self, board):
        toks = []
        for rank in range(7, -1, -1):
            for file in range(8):
                piece = board.piece_at(chess.square(file, rank))
                toks.append(self.PIECE_MAP[piece.symbol()] if piece else 0)
        return np.array(toks, dtype=np.int64)

    @torch.no_grad()
    def __call__(self, fens):
        if not fens:
            return []
        self.calls += 1
        x = torch.from_numpy(np.array([self._tokens(chess.Board(f)) for f in fens])).to(DEVICE)
        return self.model(x).squeeze(1).cpu().numpy().tolist()


# ── MCTS: engine/mcts.py, with the policy chosen per node (root vs deeper) ──
class Node:
    __slots__ = ("board", "moves", "parent", "children", "N", "W", "P")

    def __init__(self, board, moves, parent=None):
        self.board, self.moves, self.parent = board, moves, parent
        self.children = {}
        self.N, self.W, self.P = 0, 0.0, 0.0


class MCTS:
    """Line-for-line the production search (selection, expansion, evaluation, backup, move choice).
    The only change: expansion asks `root_policy` when the node is the root and `deep_policy`
    otherwise. With both equal to the base policy this is the production engine."""

    def __init__(self, moves, board, value_fn, root_policy, deep_policy, sims,
                 c_punc=C_PUNC, k=TOP_K, tau=TAU):
        self.root = Node(board.copy(), tuple(moves))
        self.value_fn, self.root_policy, self.deep_policy = value_fn, root_policy, deep_policy
        self.sims, self.c_punc, self.k, self.tau = sims, c_punc, k, tau
        self.expansions = 0
        # draw avoidance: root FEN and the last 4 (fen before the move, move) pairs
        self.root_fen = board.fen()
        replay, pairs = chess.Board(), []
        for i, u in enumerate(moves):
            if i >= len(moves) - 4:
                pairs.append((replay.fen(), u))
            replay.push_uci(u)
        self.recent_pairs = pairs

    def search(self):
        for i in range(self.sims):
            node = self._select(self.root)
            value = self._evaluate(node)
            if i < self.sims - 1 and not node.board.is_game_over():
                self._expand(node)
            self._backpropagate(node, value)

        best_move, best_score = None, -float("inf")
        for key, child in self.root.children.items():
            score = child.N
            if (self.root_fen, key) in self.recent_pairs:
                score *= 0.95
            if score > best_score:
                best_score, best_move = score, key
        return best_move, best_score

    def _expand(self, node):
        policy = self.root_policy if node is self.root else self.deep_policy
        cand = policy(node.moves, node.board, self.k)
        if not cand:
            return
        self.expansions += 1
        logits = torch.tensor([s for s, _ in cand], dtype=torch.float32)
        probs = F.softmax(logits / self.tau, dim=0).tolist()

        fens, kids = [], []
        for prob, (_, mv) in zip(probs, cand):
            cb = node.board.copy()
            cb.push_uci(mv)
            fens.append(cb.mirror().fen() if cb.turn == chess.BLACK else cb.fen())
            kids.append((mv, prob, cb))
        values = self.value_fn(fens)
        for (mv, prob, cb), v in zip(kids, values):
            child = Node(cb, node.moves + (mv,), parent=node)
            child.P = prob
            child.W = -v
            node.children[mv] = child

    def _select(self, node):
        c = self.c_punc
        while node.children:
            best_score, best_child = -float("inf"), None
            sqrt_N = node.N ** 0.5
            for child in node.children.values():
                Q = child.W / child.N if child.N > 0 else child.W
                U = c * child.P * sqrt_N / (child.N + 1)
                if Q + U > best_score:
                    best_score, best_child = Q + U, child
            node = best_child
        return node

    def _evaluate(self, node):
        if node.board.is_game_over():
            return 1.03 if node.board.is_checkmate() else 0
        return node.W

    def _backpropagate(self, node, value):
        while node:
            node.N += 1
            node.W += value
            value = -value
            node = node.parent


class Engine:
    def __init__(self, name, sims, root_policy, deep_policy, value_fn):
        self.name, self.sims, self.value_fn = name, sims, value_fn
        self.root_policy, self.deep_policy = root_policy, deep_policy
        self.policies = {root_policy, deep_policy}

    def choose(self, moves, board, sims=None):
        """Returns (uci move, stats). Timing covers the whole search."""
        before = {p.name: p.calls for p in self.policies}
        v0 = self.value_fn.calls
        t0 = time.perf_counter()
        mcts = MCTS(moves, board, self.value_fn, self.root_policy, self.deep_policy, sims or self.sims)
        move, _ = mcts.search()
        dt = time.perf_counter() - t0
        calls = {p.name: p.calls - before[p.name] for p in self.policies}
        return move, dict(sec=dt, base_calls=calls.get("base", 0), side_calls=calls.get("sidekick", 0),
                          value_calls=self.value_fn.calls - v0, expansions=mcts.expansions)


# ── Openings ──
def opening_board(line):
    board, moves = chess.Board(), []
    for san in line.split():
        moves.append(board.push_san(san).uci())
    return board, tuple(moves)


# ── Checks before spending hours ──
def selftest(prod, sidekick, base_policy, side_policy, value_fn):
    board, moves = opening_board(BOOK[0])
    for fn in (base_policy, side_policy):
        out = fn(moves, board, TOP_K)
        assert len(out) == TOP_K and all(a[0] >= b[0] for a, b in zip(out, out[1:]))
        assert all(chess.Move.from_uci(u) in board.legal_moves for _, u in out)
        print(f"  {fn.name:9s} top-3 after '{BOOK[0]}':", [(round(s, 2), u) for s, u in out[:3]])

    m1, _ = prod.choose(moves, board, sims=60)
    m2, _ = prod.choose(moves, board, sims=60)
    assert m1 == m2, "search is not deterministic on this machine"          # 1. determinism

    same = Engine("hybrid-with-base", 60, base_policy, base_policy, value_fn)
    assert same.choose(moves, board)[0] == m1                                # 2. hybrid path == prod

    _, st = sidekick.choose(moves, board, sims=60)                           # 3. large policy once per move
    assert st["base_calls"] == 1 and st["side_calls"] == st["expansions"] - 1, st
    print("  sidekick engine, 60 sims:", st)
    print("  self-test OK")


# ── Calibration: sidekick simulations so its time per move matches prod's ──
def mean_time(engine, positions, sims):
    return float(np.mean([engine.choose(m, b, sims=sims)[1]["sec"] for b, m in positions]))


def calibrate(prod, sidekick, prod_sims):
    positions = [(chess.Board(), ())] + [opening_board(l) for l in BOOK[:CALIB_POSITIONS - 1]]
    mean_time(prod, positions[:2], prod_sims)         # warm-up
    mean_time(sidekick, positions[:2], prod_sims)
    t_prod = mean_time(prod, positions, prod_sims)
    t_same = mean_time(sidekick, positions, prod_sims)
    print(f"  prod     @ {prod_sims} sims: {t_prod:.3f} s/move")
    print(f"  sidekick @ {prod_sims} sims: {t_same:.3f} s/move  (x{t_prod / t_same:.2f} faster)")
    cap = MAX_SIM_MULT * prod_sims
    sims = int(round(min(prod_sims * t_prod / t_same, cap)))
    for _ in range(2):                                # cost is close to, not exactly, linear in sims
        t = mean_time(sidekick, positions, sims)
        print(f"  sidekick @ {sims} sims: {t:.3f} s/move")
        sims = int(round(min(sims * t_prod / t, cap)))
    print(f"  => equal-time sidekick simulations: {sims} (prod: {prod_sims})")
    return sims, t_prod


# ── Play ──
def play_game(line, white, black):
    board, moves = opening_board(line)
    moves = list(moves)
    n_book = len(moves)
    per_engine = {white.name: [], black.name: []}
    while not board.is_game_over(claim_draw=True) and len(moves) < MAX_PLIES:
        eng = white if board.turn == chess.WHITE else black
        mv, st = eng.choose(tuple(moves), board)
        if mv is None:
            break
        per_engine[eng.name].append(st)
        board.push_uci(mv)
        moves.append(mv)
    over = board.is_game_over(claim_draw=True)
    result = board.result(claim_draw=True) if over else "1/2-1/2"
    pgn = chess.pgn.Game.from_board(board)
    pgn.headers.update(White=white.name, Black=black.name, Result=result,
                       Event="sidekick head-to-head", Opening=line or "start")
    if not over:
        pgn.headers["Termination"] = "adjudicated: ply cap"
    return dict(opening=line, white=white.name, black=black.name, result=result, plies=len(moves),
                book_plies=n_book, per_engine=per_engine, pgn=str(pgn), adjudicated=not over)


def run_matchup(name, prod, side, openings, prod_sims, deadline=None):
    """Plays every opening with both colours. With a deadline (time.time()), no new opening is
    started after it, so the games played always come in complete colour pairs."""
    games = []
    bar = tqdm(total=2 * len(openings), desc=name)
    for line in openings:
        if deadline is not None and time.time() > deadline:
            print(f"  time budget reached: stopping {name} after {len(games)} games")
            break
        for white, black in ((side, prod), (prod, side)):
            g = play_game(line, white, black)
            games.append(g)
            bar.update(1)
            bar.set_postfix(last=f"{g['white']}-{g['black']} {g['result']}", plies=g["plies"])
        with open(os.path.join(OUT_DIR, f"h2h_{name}.json"), "w") as f:
            json.dump(dict(name=name, side_sims=side.sims, prod_sims=prod_sims, games=games), f)
        with open(os.path.join(OUT_DIR, f"h2h_{name}.pgn"), "w") as f:
            f.write("\n\n".join(g["pgn"] for g in games) + "\n")
    bar.close()
    return games


# ── Report ──
def score_for(g, who):
    if g["result"] == "1/2-1/2":
        return 0.5
    return float((g["result"] == "1-0") == (g["white"] == who))


def elo(s):
    s = min(max(s, 1e-3), 1 - 1e-3)
    return -400 * math.log10(1 / s - 1)


def report(results, prod_sims):
    rng = np.random.default_rng(0)
    summary = {}
    for name, (side_sims, games) in results.items():
        if not games:
            print(f"\n=== {name}: no games finished ===")
            continue
        sc = np.array([score_for(g, "sidekick") for g in games])
        w, d, l = int((sc == 1).sum()), int((sc == 0.5).sum()), int((sc == 0).sum())
        pair = np.array([sc[i:i + 2].mean() for i in range(0, len(sc), 2)])   # one value per opening
        boots = np.array([rng.choice(pair, len(pair)).mean() for _ in range(5000)])
        lo, hi = np.percentile(boots, [2.5, 97.5])
        print(f"\n=== {name}: sidekick @ {side_sims} sims vs prod @ {prod_sims} sims ({len(games)} games) ===")
        print(f"sidekick: +{w} ={d} -{l}   score {sc.mean():.3f}  (95% CI {lo:.3f} to {hi:.3f}, bootstrap over openings)")
        print(f"approx Elo difference (sidekick - prod): {elo(sc.mean()):+.0f}  (CI {elo(lo):+.0f} to {elo(hi):+.0f})")
        print(f"adjudicated at ply cap: {sum(g['adjudicated'] for g in games)}; "
              f"mean plies: {np.mean([g['plies'] for g in games]):.0f}")
        row = {}
        for who in ("prod", "sidekick"):
            st = [s for g in games for s in g["per_engine"][who]]
            row[who] = dict(sec_per_move=float(np.mean([s["sec"] for s in st])),
                            base_calls=float(np.mean([s["base_calls"] for s in st])),
                            side_calls=float(np.mean([s["side_calls"] for s in st])),
                            expansions=float(np.mean([s["expansions"] for s in st])), moves=len(st))
            r = row[who]
            print(f"  {who:9s} {r['sec_per_move']:.3f} s/move | per move: {r['base_calls']:.1f} base-policy calls, "
                  f"{r['side_calls']:.1f} sidekick calls, {r['expansions']:.0f} expansions ({r['moves']} moves)")
        summary[name] = dict(side_sims=side_sims, prod_sims=prod_sims, games=len(games), wins=w, draws=d,
                             losses=l, score=float(sc.mean()), ci95=[float(lo), float(hi)], engines=row)
    with open(os.path.join(OUT_DIR, "h2h_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nSaved h2h_summary.json, h2h_<matchup>.json and h2h_<matchup>.pgn to {OUT_DIR}")


In [ ]:
def main(argv=None):
    global MAX_PLIES
    ap = argparse.ArgumentParser()
    ap.add_argument("--openings", type=int, default=len(BOOK), help="book lines to use (each is played with both colours)")
    ap.add_argument("--start-only", action="store_true", help="only the initial position, both colours (2 games per matchup)")
    ap.add_argument("--matchups", choices=["equal_time", "equal_sims", "both"], default="both")
    ap.add_argument("--prod-sims", type=int, default=PROD_SIMS, help="production simulations (default 400)")
    ap.add_argument("--max-plies", type=int, default=MAX_PLIES)
    ap.add_argument("--time-budget-hours", type=float, default=None,
                    help="stop starting new openings once this much total time has passed "
                         "(shared fairly between the matchups; for session limits)")
    args = ap.parse_args(argv)

    MAX_PLIES = args.max_plies
    t_start = time.time()
    openings = [""] if args.start_only else BOOK[:args.openings]
    for line in openings:
        opening_board(line)    # fails loudly if a line is illegal
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"Device: {DEVICE}; {len(openings)} opening(s) x 2 colours = {2 * len(openings)} games per matchup")

    base_model, base_stoi = load_policy(BASE_CKPT, BASE_VOCAB, BASE_CFG)
    side_model, side_stoi = load_policy(SIDEKICK_CKPT, SIDEKICK_VOCAB, SIDEKICK_CFG)
    base_policy = PolicyFn(base_model, base_stoi, "base")
    side_policy = PolicyFn(side_model, side_stoi, "sidekick")
    value_fn = ValueFn(VALUE_CKPT)

    prod = Engine("prod", args.prod_sims, base_policy, base_policy, value_fn)
    sidekick = Engine("sidekick", args.prod_sims, base_policy, side_policy, value_fn)

    print("\nSelf-test")
    selftest(prod, sidekick, base_policy, side_policy, value_fn)

    matchups = {"equal_time": "auto", "equal_sims": args.prod_sims}
    if args.matchups != "both":
        matchups = {args.matchups: matchups[args.matchups]}

    equal_time_sims = None
    if "auto" in matchups.values():
        print("\nCalibration (equal time per move)")
        equal_time_sims, t_prod = calibrate(prod, sidekick, args.prod_sims)
        est_h = 2 * len(openings) * 80 * t_prod / 3600
        print(f"  rough estimate per matchup: {est_h:.1f} h (80-ply games, prod's {t_prod:.2f} s/move for both sides)")

    results = {}
    hard_stop = None if args.time_budget_hours is None else t_start + args.time_budget_hours * 3600
    for i, (name, s) in enumerate(matchups.items()):
        side_sims = equal_time_sims if s == "auto" else int(s)
        print(f"\n=== {name}: sidekick @ {side_sims} sims vs prod @ {args.prod_sims} sims ===")
        side = Engine("sidekick", side_sims, base_policy, side_policy, value_fn)
        # each matchup gets an equal share of whatever time is left
        deadline = None if hard_stop is None else time.time() + (hard_stop - time.time()) / (len(matchups) - i)
        results[name] = (side_sims, run_matchup(name, prod, side, openings, args.prod_sims, deadline))

    report(results, args.prod_sims)


In [ ]:
# ───────────────────────────────────────────────────────────────
# Run: the exact settings
# ───────────────────────────────────────────────────────────────
main([
    "--openings", "10",            # 10 book lines x 2 colours = 20 games per matchup
    "--matchups", "both",          # equal_time (calibrated simulations) and equal_sims (400 vs 400)
    "--prod-sims", "400",          # production hard mode
    "--time-budget-hours", "8.0",  # Kaggle GPU limit is 9 h; shared fairly between the matchups
])
